# Transformation and Loading Into Silver

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS insurance_cat.silver


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
df_agents = spark.sql("select * from insurance_cat.bronze.agents")

In [0]:
df_agents.printSchema()

In [0]:
# Counting of duplicates
df_dis = df_agents.distinct().count()
df_count = df_agents.count()
dup = df_count - df_dis
print(dup)

In [0]:
from pyspark.sql.functions import col 
df_agents.filter(col("agent_id").isNull()).count()


In [0]:
# counting null values 

df_null = df_agents.select([count(when(isnull(c), c)).alias(c) for c in df_agents.columns])
display(df_null)

In [0]:
df_nulls = df_agents.where(col('agent_id').isNull())

In [0]:
df_agents.printSchema()

In [0]:
df_agents.display()

In [0]:

# Reading agent table from bronze layer

bronze_agent = spark.sql("select * from insurance_cat.bronze.agents where ingestion_date = current_date()")

#applying transformation on agent table

silver_agent = bronze_agent \
    .dropDuplicates(["agent_id"]) \
    .withColumnRenamed("is_active\r", "is_active") \
    .filter(col("agent_id").isNotNull()) \
    .withColumn("first_name",        initcap(trim(col("first_name")))) \
    .withColumn("last_name",         initcap(trim(col("last_name")))) \
    .withColumn("email",             trim(col("email"))) \
    .withColumn("phone",             regexp_replace(col("phone"), r"[\s\-\(\)]", "")) \
    .withColumn("region",            initcap(trim(col("region")))) \
    .withColumn("specialisation",    initcap(trim(col("specialisation")))) \
    .withColumn("is_active",         col("is_active").cast("boolean")) \
    .withColumn("is_current", lit(True))\
    .withColumn("hire_date",         to_date(col("hire_date"), "yyyy-MM-dd")) \
    .withColumn("commission_rate",   col("commission_rate").cast("double")) \
    .withColumn("silver_ingestion_timestamp", current_timestamp()) \
    .withColumn("silver_ingestion_date",      current_date())


# CDC Merging into the silver table

if spark.catalog.tableExists("insurance_cat.silver.agents"):
    agent_silver_table = DeltaTable.forName(spark, "insurance_cat.silver.agents")

    agent_silver_table.alias("target").merge(
        source = silver_agent.alias("source"),
        condition = "target.agent_id = source.agent_id"
    ) \
        .whenMatchedUpdate(
            set = {
                "first_name": "source.first_name",
                "last_name": "source.last_name",
                "email": "source.email",
                "phone": "source.phone",
                "region": "source.region",
                "specialisation": "source.specialisation",
                "is_active": "source.is_active",
                "hire_date": "source.hire_date",
                "commission_rate": "source.commission_rate",
                "silver_ingestion_timestamp": "source.silver_ingestion_timestamp",
                "silver_ingestion_date": "source.silver_ingestion_date"
            }
        ) \
        .whenNotMatchedInsertAll()\
        .execute()
else:
    silver_agent.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("insurance_cat.silver.agents")
    

In [0]:
silver_agent.printSchema()

In [0]:
bronze_policies = spark.sql("select * from insurance_cat.bronze.policies where ingestion_date = current_date()")

In [0]:
bronze_policies.printSchema()

In [0]:
%sql
drop table insurance_cat.silver.agents

In [0]:

# Reading policies table from bronze layer

bronze_policies = spark.sql("select * from insurance_cat.bronze.policies where ingestion_date = current_date()")

#applying transformation on policies table

silver_policies = bronze_policies\
    .dropDuplicates(["policy_id"])\
    .withColumnRenamed("status\r", "status")\
    .filter(col("policy_id").isNotNull())\
    .withColumn("customer_id",        trim(col("customer_id"))) \
    .withColumn("agent_id",             trim(col("agent_id"))) \
    .withColumn("insurance_type",       trim(col("insurance_type"))) \
    .withColumn("policy_subtype",       trim(col("policy_subtype"))) \
    .withColumn("premium_amount",     col("premium_amount").cast("double")) \
    .withColumn("coverage_amount",   col("coverage_amount").cast("double")) \
    .withColumn("status", trim(col("status")))\
    .withColumn("is_current", lit(True))\
    .withColumn("payment_frequency", trim(col("payment_frequency")))\
    .withColumn("start_date",         to_date(col("start_date"), "yyyy-MM-dd")) \
    .withColumn("end_date",           to_date(col("end_date"), "yyyy-MM-dd")) \
    .withColumn("silver_ingestion_timestamp", current_timestamp()) \
    .withColumn("silver_ingestion_date",      current_date())


# CDC Merging into the silver table

if spark.catalog.tableExists("insurance_cat.silver.policies"):
    policies_silver_table = DeltaTable.forName(spark, "insurance_cat.silver.policies")

    policies_silver_table.alias("target").merge(
        source = silver_policies.alias("source"),
        condition = "target.policy_id = source.policy_id"
    ) \
        .whenMatchedUpdate(
            set = {
                "customer_id": "source.customer_id",
                "agent_id": "source.agent_id",
                "insurance_type": "source.insurance_type",
                "policy_subtype": "source.policy_subtype",
                "start_date": "source.start_date",
                "end_date": "source.end_date",
                "premium_amount": "source.premium_amount",
                "payment_frequency": "source.payment_frequency",
                "coverage_amount": "source.coverage_amount",
                "status": "source.status",
                "silver_ingestion_timestamp": "source.silver_ingestion_timestamp",
                "silver_ingestion_date": "source.silver_ingestion_date"
            }
        ) \
        .whenNotMatchedInsertAll()\
        .execute()
else:
    silver_policies.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("insurance_cat.silver.policies")


In [0]:
bronze_payments = spark.sql("select * from insurance_cat.bronze.payments where ingestion_date = current_date()")

In [0]:
bronze_payments.printSchema()

In [0]:
# Reading payments table from bronze layer

bronze_payments = spark.sql("select * from insurance_cat.bronze.payments where ingestion_date = current_date()")

#applying transformation on payments table

silver_payments = bronze_payments\
    .dropDuplicates(["payment_id"])\
    .withColumnRenamed("is_late\r", "is_late")\
    .filter(col("payment_id").isNotNull())\
    .withColumn("customer_id",        trim(col("customer_id"))) \
    .withColumn("policy_id",            trim(col("policy_id"))) \
    .withColumn("payment_date",       to_date(col("payment_date"), "yyyy-MM-dd")) \
    .withColumn("due_date",           to_date(col("due_date"), "yyyy-MM-dd")) \
    .withColumn("amount",             col("amount").cast("double")) \
    .withColumn("payment_method",     trim(col("payment_method"))) \
    .withColumn("is_current",          lit(True))\
    .withColumn("reference",          trim(col("reference")))\
    .withColumn("status", trim(col("status")))\
    .withColumn("is_late", trim(col("is_late")))\
    .withColumn("silver_ingestion_timestamp", current_timestamp()) \
    .withColumn("silver_ingestion_date",      current_date())


# CDC Merging into the silver table

if spark.catalog.tableExists("insurance_cat.silver.payments"):
    payments_silver_table = DeltaTable.forName(spark, "insurance_cat.silver.payments")
    payments_silver_table.alias("target")\
        .merge(
            source = silver_payments.alias("src"),
            condition = "target.payment_id = src.payment_id"
        )\
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll()\
        .execute()
else:
    silver_payments.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("insurance_cat.silver.payments")

In [0]:
from pyspark.sql.functions import md5, concat_ws, col, trim, to_date, regexp_replace, current_timestamp, current_date
from delta.tables import DeltaTable 

# Reading and transforming customers table and masking the PII
bronze_customers = spark.sql("select * from insurance_cat.bronze.customers where ingestion_date = current_date()")

silver_customers = bronze_customers\
    .dropDuplicates(["customer_id"])\
    .withColumnRenamed("is_active\r", "is_active")\
    .filter(col("customer_id").isNotNull())\
    .withColumn("customer_id",        trim(col("customer_id"))) \
    .withColumn("first_name",         trim(col("first_name"))) \
    .withColumn("last_name",          trim(col("last_name"))) \
    .withColumn("date_of_birth",      to_date(col("date_of_birth"), "yyyy-MM-dd")) \
    .withColumn("email",              regexp_replace(col("email"), r"^[^@]+", "***"))\
    .withColumn("phone",              regexp_replace(col("phone"), r"^\d{7}", "*******"))\
    .withColumn("address",            trim(col("address"))) \
    .withColumn("city",               trim(col("city"))) \
    .withColumn("postcode",           trim(col("postcode")))\
    .withColumn("ni_number",          regexp_replace(col("ni_number"), r"^\d{5}","*****"))\
    .withColumn("bank_account",       regexp_replace(col("bank_account"), r"^\d{5}","*****"))\
    .withColumn("is_current",          lit(True))\
    .withColumn("created_date",       to_date(col("created_date"), "yyyy-MM-dd")) \
    .withColumn("is_active",          trim(col("is_active")))\
    .withColumn("silver_ingestion_timestamp", current_timestamp()) \
    .withColumn("silver_ingestion_date",      current_date())\
    .withColumn("surrogate_key", md5(concat_ws("||", col("customer_id"), col("email"), col("phone"), col("city"), col("is_active"))))


# CDC Merging into the silver table

if spark.catalog.tableExists("insurance_cat.silver.customers"):
    customers_silver_table = DeltaTable.forName(spark, "insurance_cat.silver.customers")

    customers_silver_table.alias("target").merge(
        silver_customers.alias("source"),
        """
        target.customer_id = source.customer_id
        AND target.is_current = true
        AND target.surrogate_key != source.surrogate_key
        """
    ) \
    .whenMatchedUpdate(set={
        "is_current"          : "false",
        "effective_end_date"  : "current_date()"
    }) \
    .execute()

    print("✓ Step 1 complete — old records expired")

    # Step 2 — Insert new versions of changed records
    # and brand new records
    customers_silver_table.alias("target").merge(
        silver_customers.alias("source"),
        "target.surrogate_key = source.surrogate_key"
    ) \
    .whenNotMatchedInsertAll() \
    .execute()
else:
    silver_customers.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true")\
        .saveAsTable("insurance_cat.silver.customers")


     

In [0]:
%sql
drop table insurance_cat.silver.customers

In [0]:
bronze_coverage.printSchema()

In [0]:
bronze_coverage = spark.sql("select * from insurance_cat.bronze.coverages where ingestion_date = current_date()")
bronze_coverage.display()

In [0]:
# Reading payments table from bronze layer

bronze_risks = spark.sql("select * from insurance_cat.bronze.risk_assessments where ingestion_date = current_date()")

#applying transformation on payments table

silver_risks = bronze_risks\
    .dropDuplicates(["risk_id"])\
    .withColumnRenamed("assessed_by_agent_id\r", "assessed_by_agent_id")\
    .filter(col("risk_id").isNotNull())\
    .withColumn("risk_id",            trim(col("risk_id"))) \
    .withColumn("customer_id",        trim(col("customer_id"))) \
    .withColumn("assessment_date", to_date(col("assessment_date"), "yyyy-MM-dd")) \
    .withColumn("previous_claims_count",   col("previous_claims_count").cast("int")) \
    .withColumn("credit_score",   col("credit_score").cast("double")) \
    .withColumn("annual_income",   col("annual_income").cast("double")) \
    .withColumn("risk_score",         col("risk_score").cast("double")) \
    .withColumn("risk_level",         trim(col("risk_level"))) \
    .withColumn("is_current",         lit(True))\
    .withColumn("silver_ingestion_timestamp", current_timestamp()) \
    .withColumn("silver_ingestion_date",      current_date())\
    .withColumn("surrogate_key", md5(concat_ws("||", col("risk_id"), col("customer_id"), col("assessment_date"), col("risk_level"))))

# CDC Merging into the silver table

if spark.catalog.tableExists("insurance_cat.silver.risks"):
    risks_silver_table = DeltaTable.forName(spark, "insurance_cat.silver.risks")

    risks_silver_table.alias("target").merge(
        silver_risks.alias("source"),
        """
        target.risk_id = source.risk_id
        AND target.is_current = true
        AND target.surrogate_key != source.surrogate_key
        """
    ) \
    .whenMatchedUpdate(set={
        "is_current"          : "false",
        "effective_end_date"  : "current_date()"
    }) \
    .execute()

    print("✓ Step 1 complete — old records expired")

    # Step 2 — Insert new versions of changed records
    # and brand new records
    risks_silver_table.alias("target").merge(
        silver_risks.alias("source"),
        "target.surrogate_key = source.surrogate_key"
    ) \
    .whenNotMatchedInsertAll() \
    .execute()
else:
    silver_risks.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true")\
        .saveAsTable("insurance_cat.silver.risks")


In [0]:
%sql
drop table insurance_cat.silver.claims

In [0]:
from pyspark.sql.functions import to_date, col, current_timestamp, current_date, lit, md5, concat_ws# Reading claim table from bronze layer


bronze_claims = spark.sql("select * from insurance_cat.bronze.claims where ingestion_date = date_sub(current_date(),1)")

#applying transformation on claims table

silver_cliams = bronze_claims\
    .dropDuplicates(["claim_id"])\
    .withColumnRenamed("description\r", "description")\
    .withColumn("claim_date", to_date(col("claim_date"), "yyyy-MM-dd"))\
    .withColumn("claim_amount", col("claim_amount").try_cast("double"))\
    .withColumn("approved_amount", col("approved_amount").try_cast("double"))\
    .withColumn("settlement_date", to_date(col("settlement_date"), "yyyy-MM-dd"))\
    .filter(col("claim_amount") < 0)\
    .withColumn("is_outlier", when(col("claim_amount") > 500000, True).otherwise(False))\
    .withColumn("ingestion_timestamp", current_timestamp())\
    .withColumn("is_current", lit(True)) \
    .withColumn("ingestion_date", current_date())\
    .withColumn("effective_start_date", current_date())\
    .withColumn("effective_end_date", lit(None).cast("date"))\
    .withColumn("surrogate_key", md5(concat_ws("||", col("claim_id"), col("customer_id"), col("claim_date"))))

# CDC Merging into the silver table

if spark.catalog.tableExists("insurance_cat.silver.claims"):
    claims_silver_table = DeltaTable.forName(spark, "insurance_cat.silver.claims")

    claims_silver_table.alias("target").merge(
        silver_cliams.alias("source"),
        """
        target.claim_id = source.claim_id
        AND target.is_current = true
        AND target.surrogate_key != source.surrogate_key
        """
    ) \
    .whenMatchedUpdate(set={
        "is_current"          : "false",
        "effective_end_date"  : "current_date()"
    }) \
    .execute()

    print("✓ Step 1 complete — old records expired")

    # Step 2 — Insert new versions of changed records
    # and brand new records
    claims_silver_table.alias("target").merge(
        silver_cliams.alias("source"),
        "target.surrogate_key = source.surrogate_key"
    ) \
    .whenNotMatchedInsertAll() \
    .execute()
else:
    silver_cliams.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true")\
        .saveAsTable("insurance_cat.silver.claims")


In [0]:
bronze_coverage = spark.sql("""
    SELECT * FROM insurance_cat.bronze.coverages 
    WHERE ingestion_date = date_sub(current_date(), 1)
""")

In [0]:
bronze_coverage.printSchema()

In [0]:
display(bronze_coverage)

In [0]:

# Reading coverage table from bronze layer

bronze_coverage = spark.sql("select * from insurance_cat.bronze.coverages where ingestion_date = current_date()")

#applying transformation on coverage table



silver_coverage = bronze_coverage \
    .dropDuplicates(["coverage_id"]) \
    .withColumnRenamed("no_claims_discount\r", "no_claims_discount") \
    .withColumn("deductible_amount", expr("try_cast(regexp_replace(deductible_amount, '[^0-9.]', '') AS double)"))\
     .withColumn("max_coverage_limit", expr("try_cast(regexp_replace(max_coverage_limit, '[^0-9.]', '') AS double)"))\
    .withColumn("no_claims_discount", expr("try_cast(regexp_replace (no_claims_discount, '[^0-9.]', '') AS double)")) \
    .withColumn("covers_accidental_damage", trim(col("covers_accidental_damage"))) \
    .withColumn("covers_theft", trim(col("covers_theft"))) \
    .withColumn("insurance_type", trim(col("insurance_type")))\
    .withColumn("covers_fire", trim(col("covers_fire"))) \
    .withColumn("covers_flood", trim(col("covers_flood"))) \
    .withColumn("covers_liability", trim(col("covers_liability"))) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("ingestion_date", current_date()) \
    .withColumn("is_current", lit(True)) \
    .withColumn("effective_start_date", current_date()) \
    .withColumn("effective_end_date", lit(None).cast("date")) \
    .withColumn("surrogate_key", md5(concat_ws("||", col("coverage_id"), col("policy_id"))))

# CDC Merging into the silver table

if spark.catalog.tableExists("insurance_cat.silver.coverages"):
    coverage_silver_table = DeltaTable.forName(spark, "insurance_cat.silver.coverages")

    coverage_silver_table.alias("target").merge(
        silver_coverage.alias("source"),
        """
        target.coverage_id = source.coverage_id
        AND target.is_current = true
        AND target.surrogate_key != source.surrogate_key
        """
    ) \
    .whenMatchedUpdate(set={
        "is_current"          : "false",
        "effective_end_date"  : "current_date()"
    }) \
    .execute()

    print("✓ Step 1 complete — old records expired")

    # Step 2 — Insert new versions of changed records
    # and brand new records
    coverage_silver_table.alias("target").merge(
        silver_coverage.alias("source"),
        "target.surrogate_key = source.surrogate_key"
    ) \
    .whenNotMatchedInsertAll() \
    .execute()
else:
    silver_coverage.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true")\
        .saveAsTable("insurance_cat.silver.coverages")


In [0]:
df = spark.sql("select * from insurance_cat.silver.coverages")
df.display()